In [ ]:
import hvplot.pandas  # noqa
import pandas as pd
import numpy as np, os, time
from argparse import Namespace
import tmodel as tmodel
import torch
from checkpoints import CheckpointManager
import hvplot as hv
hv.extension('bokeh')
torch.set_default_device("cpu")
use_current = True

if use_current:
    args: Namespace = tmodel.load_args()
    signal_index=args.signal
    feature_type=args.feature_type
else:
    signal_index=68
    feature_type=1
    args: Namespace = tmodel.load_args(signal_index,feature_type)
version = f"{signal_index}.{feature_type}.{args.nfeatures}"

In [ ]:
data=tmodel.get_demo_data()
signals = data['signals']
T: np.ndarray = data['times'][signal_index]
X: np.ndarray = tmodel.get_features( T, feature_type, args ).astype(np.float32)
Y: np.ndarray = signals[signal_index]
validation_split = int(0.8*X.shape[0])

model = tmodel.MultiStreamModel( args.nfeatures, args.dropout_frac, args.nstreams)
optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)

In [ ]:
checkpoints: CheckpointManager = tmodel.initialize_checkpointing( version, model, optimizer, args )

Xt: torch.Tensor = torch.from_numpy( X )
Pt: torch.Tensor = model( Xt )
P: np.ndarray = Pt.detach().numpy()
Ttrain=T[:validation_split]
Ptrain=P[:validation_split,0]
Tval=T[validation_split:]
Pval=P[validation_split:,0]

In [ ]:
title=f'Signal {args.signal} (ftype={args.feature_type}): nfeatures={args.nfeatures}'

target       = pd.DataFrame({ 't': T,      's': Y })
train_result = pd.DataFrame({ 't': Ttrain, 's': Ptrain })
val_result   = pd.DataFrame({ 't': Tval,   's': Pval })

pargs = dict( x='t', y='s', ylim=(Y.min()*.98,Y.max()*1.02) )
fargs = dict( legend_position='right', show_legend=True, title=title, xlabel='Time', height=500, width=1500 )
plot1 =       target.hvplot.line( **pargs, label='Target',     color='red'   )
plot2 = train_result.hvplot.line( **pargs, label='Train',      color='blue'  )
plot3 =   val_result.hvplot.line( **pargs, label='Validation', color='green' )

overlay_plot = ( plot1 * plot2 * plot3 ).opts( **fargs )
overlay_plot